# AirScope — Beijing PM2.5 Machine Learning Lab

Workbook praktik untuk menghubungkan materi buku *Implementasi Algoritma Machine Learning Berbasis Web dengan Framework Streamlit* (BAB 1–7) ke dataset Beijing PM2.5.

Alur: **load data → eksplorasi → pembersihan → feature engineering → split → SVM / Decision Tree / Random Forest / Naïve Bayes → evaluasi → K-Means → simpan model → jalankan Streamlit**.

> Target klasifikasi adalah label pembelajaran (Rendah ≤35, Sedang 35–75, Tinggi >75 µg/m³), bukan kategori AQI resmi. Kolom `pm2.5` hanya digunakan untuk membentuk target `y`, tidak dimasukkan ke fitur `X`.


## BAB 1 — Persiapan lingkungan dan aplikasi awal

Buat virtual environment, instal `requirements.txt`, lalu jalankan `streamlit run app.py`. Untuk contoh minimal, file `app.py` sudah menjadi aplikasi utama yang dapat dijalankan.

In [ ]:
# Jalankan dari root folder Beijing_PM25_Cupertino
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'ml_core.py').exists() and (ROOT.parent / 'ml_core.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
DATA_PATH = ROOT / 'data' / 'PRSA_data_2010.1.1-2014.12.31.csv'
print('Project:', ROOT)
print('Dataset exists:', DATA_PATH.exists())


## BAB 2 — Dasar Streamlit, teks, data, grafik, input, layout, dan control flow

Komponen BAB 2 ditunjukkan pada aplikasi `app.py`: metric cards, dataframe, tab, sidebar radio navigation, Plotly charts, sliders, selectbox, form, uploader, download button, container, dan conditional page flow.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import streamlit as st
from ml_core import (
    load_dataset, build_supervised_frame, train_classifier,
    run_kmeans, MODEL_LABELS
)

raw = load_dataset(str(DATA_PATH))
print('Shape:', raw.shape)
display(raw.head())


## BAB 2 — Eksplorasi data dan visualisasi

In [ ]:
print('Info dataset:')
raw.info()
print('\nMissing values:')
display(raw.isna().sum().to_frame('missing'))
print('\nStatistik deskriptif:')
display(raw.describe().T)

yearly = raw.groupby('year', as_index=False)['pm2.5'].mean()
fig = px.line(yearly, x='year', y='pm2.5', markers=True,
              labels={'year':'Tahun','pm2.5':'PM2.5 (µg/m³)'},
              title='Rata-rata PM2.5 per tahun')
fig.show()


## BAB 3–5 dan BAB 7 — Pembersihan, pembentukan target, dan supervised learning

- Baris dengan target PM2.5 kosong dikeluarkan karena tidak dapat diberi label.
- `No` adalah nomor baris dan tidak digunakan sebagai fitur.
- `day` tidak dipakai langsung; jam dan bulan dibuat menjadi encoding siklik.
- `cbwd` di-One-Hot Encode; fitur numerik diimputasi median dan distandardisasi.
- `pm2.5` tidak dimasukkan ke `X`, untuk menghindari target leakage.
- Default split adalah 70:30; 80:20 juga dapat dicoba mengikuti contoh BAB 7.


In [ ]:
X, y, cleaned = build_supervised_frame(raw, low_threshold=35, high_threshold=75)
print('Rows before cleaning:', len(raw))
print('Rows after dropping missing target:', len(cleaned))
print('Features:', list(X.columns))
print('Target balance:')
display(y.value_counts().rename_axis('class').to_frame('count'))
assert 'pm2.5' not in X.columns, 'Target leakage check failed'


### Train dan evaluasi empat model

In [ ]:
model_names = ['SVM', 'Decision Tree', 'Random Forest', 'Naive Bayes']
results = {}
for name in model_names:
    print('Training:', MODEL_LABELS[name])
    results[name] = train_classifier(
        name, X, y, test_size=0.30, sample_size=min(8000, len(X)),
        rf_trees=100, tree_depth=12, svm_c=1.0
    )

comparison = pd.DataFrame([
    {'model': MODEL_LABELS[name], **result['metrics']}
    for name, result in results.items()
]).sort_values('f1_weighted', ascending=False)
display(comparison)


In [ ]:
# Lihat confusion matrix dan classification report untuk satu model
chosen_name = comparison.iloc[0]['model']
chosen_key = next(k for k, v in MODEL_LABELS.items() if v == chosen_name)
chosen_result = results[chosen_key]
print('Model:', chosen_name)
print('Confusion matrix (kelas: Rendah, Sedang, Tinggi):')
print(chosen_result['confusion_matrix'])
display(pd.DataFrame(chosen_result['report']).T)
if chosen_result['feature_importance'] is not None:
    display(chosen_result['feature_importance'].head(15))


## BAB 6 — K-Means clustering

K-Means merupakan unsupervised learning, sehingga label PM2.5 tidak digunakan saat pembentukan cluster. Fitur numerik cuaca/waktu dinormalisasi, lalu rentang K=2–8 dievaluasi menggunakan inertia dan silhouette score.

In [ ]:
cluster_result = run_kmeans(raw, k=4, sample_size=3500, max_k=8)
display(cluster_result['elbow'])
display(cluster_result['profile'])
print('Silhouette score untuk K=4:', round(cluster_result['silhouette'], 4))

fig = px.scatter(cluster_result['data'], x='TEMP', y='DEWP',
                 color=cluster_result['data']['cluster'].astype(str),
                 opacity=0.6, title='K-Means: TEMP vs DEWP')
fig.show()


## Penyimpanan model (pickle) dan deployment Streamlit

Pipeline menyimpan preprocessing + estimator sehingga prediksi baru menggunakan transformasi yang sama dengan training.

In [ ]:
import pickle
models_dir = ROOT / 'models'
models_dir.mkdir(exist_ok=True)
model_filename = chosen_key.lower().replace(' ', '_') + '_pipeline.pkl'
with open(models_dir / model_filename, 'wb') as f:
    pickle.dump(chosen_result['pipeline'], f)
with open(models_dir / 'kmeans_bundle.pkl', 'wb') as f:
    pickle.dump({'model': cluster_result['model'], 'scaler': cluster_result['scaler'],
                 'features': cluster_result['features']}, f)
print('Saved to:', models_dir)


In [ ]:
# Jalankan dari terminal / VS Code pada root proyek:
# streamlit run app.py
